# Giai đoạn 3B: Tiếp tục Huấn luyện (Warm Start) BDD100k Base Model
Tài liệu này thực thi việc huấn luyện tiếp 20 Epochs (từ Epoch 20 lên 40) dựa trên trọng số `last.pt` của mẻ Dry Run trước đó.

**Lưu ý Học thuật về việc mất Optimizer State:**
Vì mẻ 20 Epoch trước đó đã **hoàn thành 100% trọn vẹn** (không bị crash giữa chừng), Ultralytics YOLO mặc định kích hoạt cơ chế `strip_optimizer` ở bước cuối cùng để giảm một nửa dung lượng file `last.pt` (từ 11MB xuống 5MB). 
Hệ lụy là toàn bộ bộ nhớ Momentum và biểu đồ Learning Rate đã bị xóa vĩnh viễn (không thể khôi phục). Do đó, cờ `resume=True` bị vô hiệu hóa.

**Giải pháp (Warm Start):**
Chúng ta sẽ nạp `last.pt` như một Base Model mới, và khởi chạy một phiên huấn luyện `model.train()` độc lập. Learning Rate sẽ được Reset lại từ đầu, tạo ra hiệu ứng **Warm Restart** (giúp mô hình bật ra khỏi cực tiểu địa phương nếu có).

In [1]:
import torch
if torch.cuda.is_available():
    print(f"Hệ thống nhận diện phần cứng xử lý: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Dung lượng VRAM khả dụng thực tế: {vram:.2f} GB")
else:
    print("Lỗi nghiêm trọng: Không phát hiện nhân tính toán CUDA. Quá trình phân tích sẽ bị gián đoạn.")

Hệ thống nhận diện phần cứng xử lý: NVIDIA GeForce RTX 3050 Laptop GPU
Dung lượng VRAM khả dụng thực tế: 4.29 GB


In [ ]:
from ultralytics import YOLO
import time
import os
import shutil

# 1. Khởi tạo bộ đếm thời gian đo lường
start_time = time.time()

# 2. Nạp trọng số từ mẻ trước (Dùng last.pt hoặc best.pt đều được vì bản chất Optimizer đã bị gọt)
model_path = 'd:/DAT301m/proposal/models/runs/bdd_day_dry_run_ep20/weights/last.pt'
print(f"[*] Đang nạp trọng số Warm Start từ: {model_path}")

model = YOLO(model_path)

# 3. Kích hoạt huấn luyện mẻ MỚI hoàn toàn
run_name = 'bdd_day_resume_ep40'

results = model.train(
    data='d:/DAT301m/proposal/data/bdd_day.yaml',
    epochs=20,             # Chạy thêm 20 epoch nữa
    patience=15,           
    batch=8,               
    cache='disk',          # Bật lại Cache vì 124GB file .npy đã CÓ SẴN trên ổ đĩa
    imgsz=640,
    device=0,
    workers=4,             
    optimizer='auto',      # Khởi động lại MuSGD
    mosaic=0.5,            
    close_mosaic=10,       
    seed=0,                
    project='d:/DAT301m/proposal/models/runs',
    name=run_name          # Tạo thư mục bdd_day_resume_ep40 độc lập
)

# 4. Đổi tên chuẩn xác và báo cáo thời gian
end_time = time.time()
total_hours = (end_time - start_time) / 3600
print(f"\n[HOÀN TẤT] Thời gian huấn luyện Warm Start (20 Epochs tiếp theo): {total_hours:.2f} giờ")

best_weight_path = f'd:/DAT301m/proposal/models/runs/{run_name}/weights/best.pt'
new_weight_path = f'd:/DAT301m/proposal/models/runs/{run_name}/weights/bdd_day_base_epoch40.pt'

if os.path.exists(best_weight_path):
    shutil.copy(best_weight_path, new_weight_path)
    print(f"[THÀNH CÔNG] Đã sao chép và đổi tên trọng số tốt nhất thành:\n{new_weight_path}")
else:
    print(f"[LỖI] Không tìm thấy file {best_weight_path}. Hãy kiểm tra lại log YOLO.")